Check the downloded train test data for ....

In [1]:
#Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
#Cell 2 — Set your corpus location
from pathlib import Path

# ============================================================
# CHANGE THIS IF REQUIRED
# ============================================================

CORPUS_ROOT = Path(
    "/content/drive/MyDrive/SLM_Biomed"
)

TRAIN_DIR = CORPUS_ROOT / "train"
TEST_DIR  = CORPUS_ROOT / "test"

AUDIT_DIR = CORPUS_ROOT / "audit"

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Corpus:", CORPUS_ROOT)
print("Train :", TRAIN_DIR)
print("Test  :", TEST_DIR)
print("Audit :", AUDIT_DIR)

Corpus: /content/drive/MyDrive/SLM_Biomed
Train : /content/drive/MyDrive/SLM_Biomed/train
Test  : /content/drive/MyDrive/SLM_Biomed/test
Audit : /content/drive/MyDrive/SLM_Biomed/audit


In [3]:
#Cell 3 — Find all shards
train_shards = sorted(
    TRAIN_DIR.glob("shard_*.jsonl.gz")
)

test_shards = sorted(
    TEST_DIR.glob("shard_*.jsonl.gz")
)

print("TRAIN SHARDS:", len(train_shards))

for p in train_shards:
    print(p.name)

print("\nTEST SHARDS:", len(test_shards))

for p in test_shards:
    print(p.name)

TRAIN SHARDS: 18
shard_00000.jsonl.gz
shard_00001.jsonl.gz
shard_00002.jsonl.gz
shard_00003.jsonl.gz
shard_00004.jsonl.gz
shard_00005.jsonl.gz
shard_00006.jsonl.gz
shard_00007.jsonl.gz
shard_00008.jsonl.gz
shard_00009.jsonl.gz
shard_00010.jsonl.gz
shard_00011.jsonl.gz
shard_00012.jsonl.gz
shard_00013.jsonl.gz
shard_00014.jsonl.gz
shard_00015.jsonl.gz
shard_00016.jsonl.gz
shard_00017.jsonl.gz

TEST SHARDS: 1
shard_00000.jsonl.gz


In [4]:
#Cell 4 — Check file sizes
def file_size_gb(path):
    return path.stat().st_size / (1024**3)


print("TRAIN")
print("=" * 60)

train_disk_size = 0

for p in train_shards:

    size = file_size_gb(p)

    train_disk_size += size

    print(
        f"{p.name:25s} "
        f"{size:8.3f} GB"
    )

print(
    f"\nTOTAL TRAIN: "
    f"{train_disk_size:.3f} GB"
)


print("\nTEST")
print("=" * 60)

test_disk_size = 0

for p in test_shards:

    size = file_size_gb(p)

    test_disk_size += size

    print(
        f"{p.name:25s} "
        f"{size:8.3f} GB"
    )

print(
    f"\nTOTAL TEST: "
    f"{test_disk_size:.3f} GB"
)


TRAIN
shard_00000.jsonl.gz         0.728 GB
shard_00001.jsonl.gz         0.363 GB
shard_00002.jsonl.gz         0.361 GB
shard_00003.jsonl.gz         0.354 GB
shard_00004.jsonl.gz         0.355 GB
shard_00005.jsonl.gz         0.355 GB
shard_00006.jsonl.gz         0.355 GB
shard_00007.jsonl.gz         0.355 GB
shard_00008.jsonl.gz         0.354 GB
shard_00009.jsonl.gz         0.356 GB
shard_00010.jsonl.gz         0.356 GB
shard_00011.jsonl.gz         0.355 GB
shard_00012.jsonl.gz         0.355 GB
shard_00013.jsonl.gz         0.355 GB
shard_00014.jsonl.gz         0.356 GB
shard_00015.jsonl.gz         0.354 GB
shard_00016.jsonl.gz         0.353 GB
shard_00017.jsonl.gz         0.212 GB

TOTAL TRAIN: 6.633 GB

TEST
shard_00000.jsonl.gz         0.178 GB

TOTAL TEST: 0.178 GB


In [5]:
#Cell 5 — Full structural audit
import gzip
import json
import time
from collections import Counter

REQUIRED_FIELDS = [
    "article_id",
    "text",
    "token_count"
]


def audit_shards(shards, split_name):

    print()
    print("=" * 80)
    print(f"{split_name.upper()} STRUCTURAL AUDIT")
    print("=" * 80)

    total_records = 0
    total_tokens = 0

    invalid_json = 0
    missing_fields = 0
    empty_text = 0
    zero_tokens = 0
    negative_tokens = 0

    duplicate_article_ids = 0

    article_ids = set()

    document_types = Counter()

    token_lengths = []

    start_time = time.time()

    for shard_number, shard_path in enumerate(shards):

        print(
            f"\n[{shard_number+1}/{len(shards)}] "
            f"{shard_path.name}"
        )

        shard_records = 0
        shard_tokens = 0

        try:

            with gzip.open(
                shard_path,
                "rt",
                encoding="utf-8"
            ) as f:

                for line_number, line in enumerate(f, 1):

                    total_records += 1
                    shard_records += 1

                    # ----------------------------------------
                    # JSON
                    # ----------------------------------------

                    try:

                        row = json.loads(line)

                    except Exception:

                        invalid_json += 1
                        continue

                    # ----------------------------------------
                    # REQUIRED FIELDS
                    # ----------------------------------------

                    if not all(
                        field in row
                        for field in REQUIRED_FIELDS
                    ):

                        missing_fields += 1

                    # ----------------------------------------
                    # TEXT
                    # ----------------------------------------

                    text = row.get("text", "")

                    if not isinstance(text, str):

                        empty_text += 1

                    elif not text.strip():

                        empty_text += 1

                    # ----------------------------------------
                    # TOKEN COUNT
                    # ----------------------------------------

                    try:

                        tokens = int(
                            row.get(
                                "token_count",
                                0
                            )
                        )

                    except Exception:

                        tokens = 0

                    if tokens == 0:
                        zero_tokens += 1

                    if tokens < 0:
                        negative_tokens += 1

                    total_tokens += tokens
                    shard_tokens += tokens

                    token_lengths.append(tokens)

                    # ----------------------------------------
                    # ARTICLE ID
                    # ----------------------------------------

                    article_id = row.get(
                        "article_id"
                    )

                    if article_id in article_ids:

                        duplicate_article_ids += 1

                    else:

                        article_ids.add(
                            article_id
                        )

                    # ----------------------------------------
                    # DOCUMENT TYPE
                    # ----------------------------------------

                    document_type = row.get(
                        "document_type",
                        "UNKNOWN"
                    )

                    document_types[
                        str(document_type)
                    ] += 1

        except Exception as e:

            print(
                f"ERROR reading {shard_path.name}:"
            )

            print(e)

            continue

        print(
            f"Records : {shard_records:,}"
        )

        print(
            f"Tokens  : {shard_tokens:,}"
        )

    elapsed = time.time() - start_time

    print("\n")
    print("=" * 80)
    print(f"{split_name.upper()} AUDIT COMPLETE")
    print("=" * 80)

    print(
        f"Records              : {total_records:,}"
    )

    print(
        f"Tokens               : {total_tokens:,}"
    )

    print(
        f"Tokens (billions)    : "
        f"{total_tokens / 1e9:.4f}B"
    )

    print(
        f"Unique article IDs   : "
        f"{len(article_ids):,}"
    )

    print(
        f"Duplicate article IDs: "
        f"{duplicate_article_ids:,}"
    )

    print(
        f"Invalid JSON         : "
        f"{invalid_json:,}"
    )

    print(
        f"Missing fields       : "
        f"{missing_fields:,}"
    )

    print(
        f"Empty text           : "
        f"{empty_text:,}"
    )

    print(
        f"Zero tokens          : "
        f"{zero_tokens:,}"
    )

    print(
        f"Negative tokens      : "
        f"{negative_tokens:,}"
    )

    print(
        f"Elapsed              : "
        f"{elapsed/60:.2f} minutes"
    )

    print("\nDocument types:")

    for k, v in document_types.most_common():

        print(
            f"{k:25s} {v:,}"
        )

    return {
        "records": total_records,
        "tokens": total_tokens,
        "article_ids": article_ids,
        "duplicate_article_ids":
            duplicate_article_ids,
        "invalid_json":
            invalid_json,
        "missing_fields":
            missing_fields,
        "empty_text":
            empty_text,
        "zero_tokens":
            zero_tokens,
        "negative_tokens":
            negative_tokens,
        "document_types":
            document_types,
        "token_lengths":
            token_lengths
    }

In [6]:
#Cell 6 — train audit
train_audit = audit_shards(
    train_shards,
    "train"
)


TRAIN STRUCTURAL AUDIT

[1/18] shard_00000.jsonl.gz
Records : 2,293,407
Tokens  : 500,000,080

[2/18] shard_00001.jsonl.gz
Records : 1,144,497
Tokens  : 250,000,277

[3/18] shard_00002.jsonl.gz
Records : 1,125,860
Tokens  : 250,000,044

[4/18] shard_00003.jsonl.gz
Records : 1,079,890
Tokens  : 250,000,137

[5/18] shard_00004.jsonl.gz
Records : 1,083,629
Tokens  : 250,000,077

[6/18] shard_00005.jsonl.gz
Records : 1,082,678
Tokens  : 250,000,226

[7/18] shard_00006.jsonl.gz
Records : 1,082,922
Tokens  : 250,000,115

[8/18] shard_00007.jsonl.gz
Records : 1,082,135
Tokens  : 250,000,378

[9/18] shard_00008.jsonl.gz
Records : 1,081,511
Tokens  : 250,000,120

[10/18] shard_00009.jsonl.gz
Records : 1,096,152
Tokens  : 250,000,397

[11/18] shard_00010.jsonl.gz
Records : 1,091,616
Tokens  : 250,000,142

[12/18] shard_00011.jsonl.gz
Records : 1,091,841
Tokens  : 250,000,067

[13/18] shard_00012.jsonl.gz
Records : 1,093,419
Tokens  : 250,000,268

[14/18] shard_00013.jsonl.gz
Records : 1,093,551

In [7]:
#Cell 7 — test audit
test_audit = audit_shards(
    test_shards,
    "test"
)


TEST STRUCTURAL AUDIT

[1/1] shard_00000.jsonl.gz
Records : 545,799
Tokens  : 124,086,229


TEST AUDIT COMPLETE
Records              : 545,799
Tokens               : 124,086,229
Tokens (billions)    : 0.1241B
Unique article IDs   : 22,839
Duplicate article IDs: 522,960
Invalid JSON         : 0
Missing fields       : 0
Empty text           : 0
Zero tokens          : 0
Negative tokens      : 0
Elapsed              : 0.17 minutes

Document types:
study                     493,992
review                    40,825
other                     10,720
clinical_case             262


In [8]:
#Cell 8 — Check train/test leakage
train_ids = train_audit["article_ids"]
test_ids = test_audit["article_ids"]

overlap = train_ids.intersection(
    test_ids
)

print("=" * 70)
print("TRAIN / TEST ARTICLE LEAKAGE")
print("=" * 70)

print(
    "Train articles:",
    f"{len(train_ids):,}"
)

print(
    "Test articles :",
    f"{len(test_ids):,}"
)

print(
    "Overlap       :",
    f"{len(overlap):,}"
)

if len(overlap) == 0:

    print(
        "\n✓ PASS — No article-level leakage detected."
    )

else:

    print(
        "\n⚠ WARNING — Train/test article overlap detected."
    )

    print(
        list(overlap)[:20]
    )


TRAIN / TEST ARTICLE LEAKAGE
Train articles: 852,935
Test articles : 22,839
Overlap       : 0

✓ PASS — No article-level leakage detected.


In [9]:
#Cell 9 — Token-length statistics
import numpy as np

train_lengths = np.array(
    train_audit["token_lengths"],
    dtype=np.int64
)

test_lengths = np.array(
    test_audit["token_lengths"],
    dtype=np.int64
)


def print_length_stats(
    lengths,
    name
):

    print()
    print("=" * 70)
    print(name)
    print("=" * 70)

    print(
        "Count:",
        f"{len(lengths):,}"
    )

    print(
        "Mean:",
        f"{np.mean(lengths):.1f}"
    )

    print(
        "Median:",
        f"{np.median(lengths):.1f}"
    )

    print(
        "P90:",
        f"{np.percentile(lengths, 90):.1f}"
    )

    print(
        "P95:",
        f"{np.percentile(lengths, 95):.1f}"
    )

    print(
        "P99:",
        f"{np.percentile(lengths, 99):.1f}"
    )

    print(
        "Maximum:",
        f"{np.max(lengths):,}"
    )


print_length_stats(
    train_lengths,
    "TRAIN TOKEN LENGTH"
)

print_length_stats(
    test_lengths,
    "TEST TOKEN LENGTH"
)


TRAIN TOKEN LENGTH
Count: 20,446,322
Mean: 227.4
Median: 181.0
P90: 390.0
P95: 503.0
P99: 953.0
Maximum: 102,267

TEST TOKEN LENGTH
Count: 545,799
Mean: 227.3
Median: 181.0
P90: 390.0
P95: 504.0
P99: 943.0
Maximum: 30,105


In [10]:
#Cell 10 -
for threshold in [
    1024,
    2048,
    4096,
    8192,
    16384
]:

    count = np.sum(
        train_lengths > threshold
    )

    percentage = (
        count /
        len(train_lengths)
        * 100
    )

    print(
        f"> {threshold:5d} tokens : "
        f"{count:,} "
        f"({percentage:.3f}%)"
    )


>  1024 tokens : 173,295 (0.848%)
>  2048 tokens : 38,051 (0.186%)
>  4096 tokens : 6,864 (0.034%)
>  8192 tokens : 1,033 (0.005%)
> 16384 tokens : 147 (0.001%)


In [11]:
#Cell 11 - — Duplicate text audit
import hashlib
import gzip
import json
from collections import Counter


def text_hash_audit(shards, name):

    print()
    print("=" * 70)
    print(f"{name.upper()} TEXT DUPLICATION")
    print("=" * 70)

    hashes = set()

    duplicates = 0
    records = 0

    for shard_path in shards:

        with gzip.open(
            shard_path,
            "rt",
            encoding="utf-8"
        ) as f:

            for line in f:

                row = json.loads(line)

                text = row.get(
                    "text",
                    ""
                )

                h = hashlib.sha256(
                    text.encode(
                        "utf-8",
                        errors="ignore"
                    )
                ).digest()

                if h in hashes:

                    duplicates += 1

                else:

                    hashes.add(h)

                records += 1

    percentage = (
        duplicates /
        max(records, 1)
        * 100
    )

    print(
        "Records:",
        f"{records:,}"
    )

    print(
        "Duplicate texts:",
        f"{duplicates:,}"
    )

    print(
        "Duplicate rate:",
        f"{percentage:.4f}%"
    )

    return {
        "records": records,
        "duplicates": duplicates,
        "duplicate_rate": percentage
    }


In [12]:
#Cell 11a
train_duplicates = text_hash_audit(
    train_shards,
    "train"
)


TRAIN TEXT DUPLICATION
Records: 20,446,322
Duplicate texts: 3,559
Duplicate rate: 0.0174%


In [13]:
#Cell 11b:
test_duplicates = text_hash_audit(
    test_shards,
    "test"
)


TEST TEXT DUPLICATION
Records: 545,799
Duplicate texts: 13
Duplicate rate: 0.0024%


In [14]:
#Cell 12 - SAve Audit Summary

import json

summary = {

    "train": {

        "records":
            train_audit["records"],

        "tokens":
            train_audit["tokens"],

        "unique_articles":
            len(train_audit["article_ids"]),

        "duplicate_article_ids":
            train_audit[
                "duplicate_article_ids"
            ],

        "invalid_json":
            train_audit[
                "invalid_json"
            ],

        "missing_fields":
            train_audit[
                "missing_fields"
            ],

        "empty_text":
            train_audit[
                "empty_text"
            ],

        "zero_tokens":
            train_audit[
                "zero_tokens"
            ],

        "negative_tokens":
            train_audit[
                "negative_tokens"
            ],

        "duplicate_text_rate":
            train_duplicates[
                "duplicate_rate"
            ]
    },

    "test": {

        "records":
            test_audit["records"],

        "tokens":
            test_audit["tokens"],

        "unique_articles":
            len(test_audit["article_ids"]),

        "duplicate_article_ids":
            test_audit[
                "duplicate_article_ids"
            ],

        "invalid_json":
            test_audit[
                "invalid_json"
            ],

        "missing_fields":
            test_audit[
                "missing_fields"
            ],

        "empty_text":
            test_audit[
                "empty_text"
            ],

        "zero_tokens":
            test_audit[
                "zero_tokens"
            ],

        "negative_tokens":
            test_audit[
                "negative_tokens"
            ],

        "duplicate_text_rate":
            test_duplicates[
                "duplicate_rate"
            ]
    },

    "train_test_article_overlap":
        len(overlap)
}


summary_file = (
    AUDIT_DIR /
    "audit_summary.json"
)


with open(
    summary_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


print(
    "Saved:",
    summary_file
)

Saved: /content/drive/MyDrive/SLM_Biomed/audit/audit_summary.json


In [15]:
#Cell 13 — Final PASS/FAIL
print()
print("=" * 80)
print("              BIOMED SLM CORPUS QUALITY GATE")
print("=" * 80)

checks = {

    "Train tokens >= 4.9B":
        train_audit["tokens"] >= 4.9e9,

    "Test tokens >= 140M":
        test_audit["tokens"] >= 140e6,

    "Invalid JSON == 0":
        train_audit["invalid_json"] == 0
        and
        test_audit["invalid_json"] == 0,

    "Missing fields == 0":
        train_audit["missing_fields"] == 0
        and
        test_audit["missing_fields"] == 0,

    "Empty text == 0":
        train_audit["empty_text"] == 0
        and
        test_audit["empty_text"] == 0,

    "Negative tokens == 0":
        train_audit["negative_tokens"] == 0
        and
        test_audit["negative_tokens"] == 0,

    "Train/Test article overlap == 0":
        len(overlap) == 0
}


for name, passed in checks.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{name}"
    )


if all(checks.values()):

    print()
    print(
        "🎉 PASS — Corpus is structurally ready "
        "for training."
    )

else:

    print()
    print(
        "⚠ REVIEW REQUIRED — One or more "
        "quality gates failed."
    )


              BIOMED SLM CORPUS QUALITY GATE
✗ Train tokens >= 4.9B
✗ Test tokens >= 140M
✓ Invalid JSON == 0
✓ Missing fields == 0
✓ Empty text == 0
✓ Negative tokens == 0
✓ Train/Test article overlap == 0

⚠ REVIEW REQUIRED — One or more quality gates failed.


In [16]:
# ============================================================
# CELL 14 — ARTICLE / RECORD CONSISTENCY AUDIT
# ============================================================

import gzip
import json
from collections import defaultdict, Counter
import numpy as np
import gc
import time

print("=" * 80)
print("ARTICLE / RECORD CONSISTENCY AUDIT")
print("=" * 80)

# ------------------------------------------------------------
# IMPORTANT
# We store only compact statistics.
# We DO NOT store all document text in RAM.
# ------------------------------------------------------------

article_record_counts = Counter()
article_token_counts = Counter()

# For detecting whether repeated records are actually different
article_text_hashes = defaultdict(set)

# Keep only a small sample of article IDs
SAMPLE_ARTICLES = 20
sample_article_ids = set()

start_time = time.time()

# ============================================================
# PASS 1 — TRAIN
# ============================================================

print("\nScanning TRAIN...")

for shard_num, shard_path in enumerate(train_shards):

    print(
        f"  [{shard_num + 1}/{len(train_shards)}] "
        f"{shard_path.name}"
    )

    with gzip.open(
        shard_path,
        "rt",
        encoding="utf-8"
    ) as f:

        for line in f:

            row = json.loads(line)

            article_id = str(
                row.get("article_id", "")
            )

            text = row.get(
                "text",
                ""
            )

            tokens = int(
                row.get(
                    "token_count",
                    0
                )
            )

            article_record_counts[
                article_id
            ] += 1

            article_token_counts[
                article_id
            ] += tokens

            # ------------------------------------------------
            # Keep text hashes only for a SMALL sample
            # ------------------------------------------------

            if (
                len(sample_article_ids)
                < SAMPLE_ARTICLES
            ):

                sample_article_ids.add(
                    article_id
                )

            if article_id in sample_article_ids:

                import hashlib

                h = hashlib.sha256(
                    text.encode(
                        "utf-8",
                        errors="ignore"
                    )
                ).hexdigest()

                article_text_hashes[
                    article_id
                ].add(h)


# ============================================================
# BASIC STATISTICS
# ============================================================

counts = np.array(
    list(article_record_counts.values()),
    dtype=np.int64
)

tokens = np.array(
    list(article_token_counts.values()),
    dtype=np.int64
)

print("\n")
print("=" * 80)
print("TRAIN ARTICLE STATISTICS")
print("=" * 80)

print(
    "Unique articles:",
    f"{len(counts):,}"
)

print(
    "Total records:",
    f"{sum(counts):,}"
)

print(
    "Mean records/article:",
    f"{np.mean(counts):.2f}"
)

print(
    "Median records/article:",
    f"{np.median(counts):.0f}"
)

print(
    "P90 records/article:",
    f"{np.percentile(counts, 90):.0f}"
)

print(
    "P95 records/article:",
    f"{np.percentile(counts, 95):.0f}"
)

print(
    "P99 records/article:",
    f"{np.percentile(counts, 99):.0f}"
)

print(
    "Maximum records/article:",
    f"{np.max(counts):,}"
)


# ============================================================
# DISTRIBUTION
# ============================================================

print("\n")
print("=" * 80)
print("RECORDS PER ARTICLE DISTRIBUTION")
print("=" * 80)

for threshold in [
    1,
    2,
    5,
    10,
    20,
    50,
    100,
    500,
    1000
]:

    n = np.sum(
        counts >= threshold
    )

    pct = (
        n /
        len(counts)
        * 100
    )

    print(
        f">= {threshold:4d} records/article : "
        f"{n:10,} articles "
        f"({pct:7.3f}%)"
    )


# ============================================================
# TOP 30 ARTICLES BY RECORD COUNT
# ============================================================

print("\n")
print("=" * 80)
print("TOP 30 ARTICLES BY NUMBER OF RECORDS")
print("=" * 80)

top_articles = (
    article_record_counts
    .most_common(30)
)

for rank, (
    article_id,
    record_count
) in enumerate(
    top_articles,
    1
):

    total_article_tokens = (
        article_token_counts[
            article_id
        ]
    )

    print(
        f"{rank:2d}. "
        f"ID={article_id} | "
        f"records={record_count:,} | "
        f"tokens={total_article_tokens:,}"
    )


# ============================================================
# SAMPLE ARTICLE INSPECTION
# ============================================================

print("\n")
print("=" * 80)
print("SAMPLE ARTICLE RECORD CONSISTENCY")
print("=" * 80)

for article_id in list(
    sample_article_ids
)[:20]:

    record_count = (
        article_record_counts[
            article_id
        ]
    )

    unique_texts = len(
        article_text_hashes.get(
            article_id,
            set()
        )
    )

    total_article_tokens = (
        article_token_counts[
            article_id
        ]
    )

    print(
        f"Article {article_id}"
    )

    print(
        f"  Records       : "
        f"{record_count:,}"
    )

    print(
        f"  Total tokens  : "
        f"{total_article_tokens:,}"
    )

    print(
        f"  Unique texts  : "
        f"{unique_texts:,}"
    )

    if record_count == unique_texts:

        print(
            "  Interpretation: "
            "records appear text-distinct"
        )

    elif unique_texts == 1:

        print(
            "  WARNING: all records "
            "have identical text"
        )

    else:

        print(
            "  Interpretation: "
            "some text repetition exists"
        )


# ============================================================
# FINAL INTERPRETATION
# ============================================================

elapsed = time.time() - start_time

print("\n")
print("=" * 80)
print("AUDIT COMPLETE")
print("=" * 80)

print(
    f"Elapsed: {elapsed / 60:.2f} minutes"
)

print()
print(
    "IMPORTANT:"
)

print(
    "High records/article is NOT automatically "
    "a duplication problem."
)

print(
    "If repeated article IDs correspond to "
    "different text chunks, this is expected."
)

print(
    "The next step is to inspect the output "
    "and determine whether these are legitimate "
    "article chunks."
)

ARTICLE / RECORD CONSISTENCY AUDIT

Scanning TRAIN...
  [1/18] shard_00000.jsonl.gz
  [2/18] shard_00001.jsonl.gz
  [3/18] shard_00002.jsonl.gz
  [4/18] shard_00003.jsonl.gz
  [5/18] shard_00004.jsonl.gz
  [6/18] shard_00005.jsonl.gz
  [7/18] shard_00006.jsonl.gz
  [8/18] shard_00007.jsonl.gz
  [9/18] shard_00008.jsonl.gz
  [10/18] shard_00009.jsonl.gz
  [11/18] shard_00010.jsonl.gz
  [12/18] shard_00011.jsonl.gz
  [13/18] shard_00012.jsonl.gz
  [14/18] shard_00013.jsonl.gz
  [15/18] shard_00014.jsonl.gz
  [16/18] shard_00015.jsonl.gz
  [17/18] shard_00016.jsonl.gz
  [18/18] shard_00017.jsonl.gz


TRAIN ARTICLE STATISTICS
Unique articles: 852,935
Total records: 20,446,322
Mean records/article: 23.97
Median records/article: 23
P90 records/article: 42
P95 records/article: 50
P99 records/article: 70
Maximum records/article: 1,810


RECORDS PER ARTICLE DISTRIBUTION
>=    1 records/article :    852,935 articles (100.000%)
>=    2 records/article :    835,570 articles ( 97.964%)
>=    5 reco

In [17]:
# ============================================================
# CELL 15 — REPRESENTATIVE CONTENT QUALITY AUDIT
# ============================================================

import gzip
import json
import random
import re
import math
import time
import hashlib
from pathlib import Path
from collections import Counter, defaultdict

print("=" * 80)
print("REPRESENTATIVE CONTENT QUALITY AUDIT")
print("=" * 80)

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

TARGET_SAMPLES = 2000
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

# Output directory
CONTENT_AUDIT_DIR = AUDIT_DIR / "content_audit"
CONTENT_AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Target samples :", TARGET_SAMPLES)
print("Output folder  :", CONTENT_AUDIT_DIR)


# ============================================================
# PART 1
# FIRST PASS:
# Collect lightweight metadata for every record
# ============================================================

print("\n")
print("=" * 80)
print("PASS 1 — COLLECTING RECORD METADATA")
print("=" * 80)

metadata = []

start_time = time.time()

for shard_num, shard_path in enumerate(train_shards):

    print(
        f"[{shard_num + 1}/{len(train_shards)}] "
        f"{shard_path.name}"
    )

    with gzip.open(
        shard_path,
        "rt",
        encoding="utf-8"
    ) as f:

        for line_number, line in enumerate(f):

            try:
                row = json.loads(line)
            except Exception:
                continue

            text = str(
                row.get("text", "")
            )

            article_id = str(
                row.get(
                    "article_id",
                    ""
                )
            )

            token_count = int(
                row.get(
                    "token_count",
                    0
                )
            )

            document_type = str(
                row.get(
                    "document_type",
                    "UNKNOWN"
                )
            )

            metadata.append({
                "shard": shard_num,
                "shard_name": shard_path.name,
                "line": line_number,
                "article_id": article_id,
                "document_type": document_type,
                "token_count": token_count,
                "text_length": len(text)
            })


elapsed = time.time() - start_time

print()
print(
    f"Metadata collected: "
    f"{len(metadata):,} records"
)

print(
    f"Elapsed: {elapsed / 60:.2f} minutes"
)


# ============================================================
# PART 2
# CREATE STRATIFICATION VARIABLES
# ============================================================

print("\n")
print("=" * 80)
print("PASS 2 — BUILDING STRATA")
print("=" * 80)


def length_bucket(tokens):

    if tokens < 128:
        return "short_<128"

    elif tokens < 512:
        return "medium_128_511"

    elif tokens < 1024:
        return "long_512_1023"

    elif tokens < 2048:
        return "very_long_1024_2047"

    else:
        return "extreme_>=2048"


for item in metadata:

    item["length_bucket"] = length_bucket(
        item["token_count"]
    )


# ============================================================
# PART 3
# CREATE STRATA
#
# We combine:
#   shard
#   document type
#   token length
#
# ============================================================

strata = defaultdict(list)

for idx, item in enumerate(metadata):

    key = (
        item["shard"],
        item["document_type"],
        item["length_bucket"]
    )

    strata[key].append(idx)


print(
    "Number of strata:",
    len(strata)
)


# ============================================================
# PART 4
# STRATIFIED SAMPLE
#
# Proportional sampling across strata,
# while guaranteeing representation from
# all available shards.
# ============================================================

print("\n")
print("=" * 80)
print("PASS 3 — SELECTING ~2,000 REPRESENTATIVE RECORDS")
print("=" * 80)

total_records = len(metadata)

# Desired proportional sample
desired = {}

for key, indices in strata.items():

    proportion = (
        len(indices) /
        total_records
    )

    desired[key] = max(
        1,
        round(
            proportion *
            TARGET_SAMPLES
        )
    )


# Adjust total sample size
# ------------------------------------------------------------

selected_indices = set()

# First guarantee at least one record per shard
shard_to_indices = defaultdict(list)

for idx, item in enumerate(metadata):

    shard_to_indices[
        item["shard"]
    ].append(idx)


for shard, indices in shard_to_indices.items():

    selected_indices.add(
        random.choice(indices)
    )


# Then proportional selection
for key, indices in strata.items():

    n = desired[key]

    n = min(
        n,
        len(indices)
    )

    chosen = random.sample(
        indices,
        n
    )

    selected_indices.update(
        chosen
    )


# If > TARGET, randomly reduce
if len(selected_indices) > TARGET_SAMPLES:

    selected_indices = set(
        random.sample(
            list(selected_indices),
            TARGET_SAMPLES
        )
    )


# If < TARGET, fill randomly
if len(selected_indices) < TARGET_SAMPLES:

    remaining = list(
        set(range(total_records))
        -
        selected_indices
    )

    needed = (
        TARGET_SAMPLES
        -
        len(selected_indices)
    )

    selected_indices.update(
        random.sample(
            remaining,
            min(
                needed,
                len(remaining)
            )
        )
    )


selected_indices = sorted(
    selected_indices
)

print(
    "Selected records:",
    len(selected_indices)
)


# ============================================================
# PART 5
# RETRIEVE THE ACTUAL TEXT
#
# Only ~2,000 records are kept.
# ============================================================

print("\n")
print("=" * 80)
print("PASS 4 — RETRIEVING SELECTED TEXT")
print("=" * 80)

selected_lookup = {
    (
        metadata[i]["shard"],
        metadata[i]["line"]
    ): i
    for i in selected_indices
}


samples = []

for shard_num, shard_path in enumerate(train_shards):

    wanted = [
        key
        for key in selected_lookup
        if key[0] == shard_num
    ]

    if not wanted:
        continue

    wanted_lines = {
        key[1]
        for key in wanted
    }

    print(
        f"Reading selected records from "
        f"{shard_path.name}"
    )

    with gzip.open(
        shard_path,
        "rt",
        encoding="utf-8"
    ) as f:

        for line_number, line in enumerate(f):

            if line_number not in wanted_lines:
                continue

            try:
                row = json.loads(line)
            except Exception:
                continue

            metadata_index = selected_lookup[
                (shard_num, line_number)
            ]

            samples.append({
                "sample_index":
                    metadata_index,

                "shard":
                    shard_path.name,

                "line":
                    line_number,

                "article_id":
                    str(
                        row.get(
                            "article_id",
                            ""
                        )
                    ),

                "document_type":
                    str(
                        row.get(
                            "document_type",
                            "UNKNOWN"
                        )
                    ),

                "token_count":
                    int(
                        row.get(
                            "token_count",
                            0
                        )
                    ),

                "text":
                    str(
                        row.get(
                            "text",
                            ""
                        )
                    )
            })


print(
    "\nRetrieved:",
    len(samples),
    "records"
)


# ============================================================
# PART 6
# AUTOMATED QUALITY CHECKS
# ============================================================

print("\n")
print("=" * 80)
print("PASS 5 — AUTOMATED CONTENT CHECKS")
print("=" * 80)


# ------------------------------------------------------------
# Patterns
# ------------------------------------------------------------

HTML_PATTERN = re.compile(
    r"<[^>]+>"
)

XML_PATTERN = re.compile(
    r"</?[A-Za-z][^>]*>"
)

URL_PATTERN = re.compile(
    r"https?://|www\.",
    re.IGNORECASE
)

EMAIL_PATTERN = re.compile(
    r"\b[A-Za-z0-9._%+-]+@"
    r"[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
)

REFERENCE_PATTERN = re.compile(
    r"\b(references|bibliography|"
    r"acknowledg(e)?ments)\b",
    re.IGNORECASE
)

FIGURE_PATTERN = re.compile(
    r"\b(fig\.?|figure|table)\s*\d+",
    re.IGNORECASE
)

DOI_PATTERN = re.compile(
    r"\b10\.\d{4,9}/[-._;()/:A-Z0-9]+\b",
    re.IGNORECASE
)

MULTI_SPACE_PATTERN = re.compile(
    r"[ \t]{5,}"
)

BAD_UNICODE_PATTERN = re.compile(
    r"[\ufffd]"
)


# ------------------------------------------------------------
# Counters
# ------------------------------------------------------------

quality_counts = Counter()

quality_by_type = defaultdict(
    Counter
)

quality_by_shard = defaultdict(
    Counter
)

token_lengths = []


def check_record(sample):

    text = sample["text"]

    flags = []

    # --------------------------------------------------------
    # Empty
    # --------------------------------------------------------

    if not text.strip():

        flags.append(
            "EMPTY"
        )

    # --------------------------------------------------------
    # Very short
    # --------------------------------------------------------

    if len(text.strip()) < 100:

        flags.append(
            "VERY_SHORT_TEXT"
        )

    # --------------------------------------------------------
    # HTML / XML
    # --------------------------------------------------------

    html_matches = len(
        HTML_PATTERN.findall(text)
    )

    xml_matches = len(
        XML_PATTERN.findall(text)
    )

    if html_matches >= 5:

        flags.append(
            "HEAVY_HTML"
        )

    elif html_matches > 0:

        flags.append(
            "HTML_PRESENT"
        )

    if xml_matches >= 5:

        flags.append(
            "HEAVY_XML"
        )

    # --------------------------------------------------------
    # URLs / emails
    # --------------------------------------------------------

    if URL_PATTERN.search(text):

        flags.append(
            "URL_PRESENT"
        )

    if EMAIL_PATTERN.search(text):

        flags.append(
            "EMAIL_PRESENT"
        )

    # --------------------------------------------------------
    # References
    # --------------------------------------------------------

    if REFERENCE_PATTERN.search(text):

        flags.append(
            "REFERENCE_SECTION"
        )

    # --------------------------------------------------------
    # Figure / table references
    # --------------------------------------------------------

    figure_count = len(
        FIGURE_PATTERN.findall(text)
    )

    if figure_count >= 10:

        flags.append(
            "FIGURE_TABLE_HEAVY"
        )

    # --------------------------------------------------------
    # DOI
    # --------------------------------------------------------

    doi_count = len(
        DOI_PATTERN.findall(text)
    )

    if doi_count >= 10:

        flags.append(
            "DOI_HEAVY"
        )

    # --------------------------------------------------------
    # Bad unicode
    # --------------------------------------------------------

    if BAD_UNICODE_PATTERN.search(text):

        flags.append(
            "UNICODE_REPLACEMENT"
        )

    # --------------------------------------------------------
    # Excessive whitespace
    # --------------------------------------------------------

    if MULTI_SPACE_PATTERN.search(text):

        flags.append(
            "EXCESSIVE_WHITESPACE"
        )

    # --------------------------------------------------------
    # Character statistics
    # --------------------------------------------------------

    if len(text) > 0:

        alpha_chars = sum(
            c.isalpha()
            for c in text
        )

        numeric_chars = sum(
            c.isdigit()
            for c in text
        )

        printable_chars = sum(
            c.isprintable()
            for c in text
        )

        alpha_ratio = (
            alpha_chars /
            len(text)
        )

        numeric_ratio = (
            numeric_chars /
            len(text)
        )

        printable_ratio = (
            printable_chars /
            len(text)
        )

        # Extremely unusual text
        if alpha_ratio < 0.20:

            flags.append(
                "LOW_ALPHA_CONTENT"
            )

        if printable_ratio < 0.90:

            flags.append(
                "LOW_PRINTABLE_RATIO"
            )

        if numeric_ratio > 0.40:

            flags.append(
                "NUMERIC_HEAVY"
            )

    return flags


# ------------------------------------------------------------
# Run checks
# ------------------------------------------------------------

for sample in samples:

    flags = check_record(
        sample
    )

    sample["quality_flags"] = flags

    sample["quality_flag_count"] = len(
        flags
    )

    token_lengths.append(
        sample["token_count"]
    )

    document_type = sample[
        "document_type"
    ]

    shard = sample[
        "shard"
    ]

    if len(flags) == 0:

        quality_counts[
            "CLEAN"
        ] += 1

        quality_by_type[
            document_type
        ]["CLEAN"] += 1

        quality_by_shard[
            shard
        ]["CLEAN"] += 1

    else:

        quality_counts[
            "FLAGGED"
        ] += 1

        quality_by_type[
            document_type
        ]["FLAGGED"] += 1

        quality_by_shard[
            shard
        ]["FLAGGED"] += 1

    for flag in flags:

        quality_counts[
            flag
        ] += 1

        quality_by_type[
            document_type
        ][flag] += 1

        quality_by_shard[
            shard
        ][flag] += 1


# ============================================================
# PART 7
# LANGUAGE HEURISTIC
# ============================================================

print("\n")
print("=" * 80)
print("LANGUAGE / ENGLISH-LIKENESS CHECK")
print("=" * 80)

# This is a heuristic, not a definitive language classifier.
# It checks the proportion of common English words.

COMMON_ENGLISH = {
    "the", "of", "and", "to",
    "in", "a", "for", "is",
    "on", "that", "by", "with",
    "as", "was", "are", "from",
    "this", "or", "an", "be",
    "which", "these", "were",
    "at", "it", "have", "has",
    "not", "but", "their",
    "they", "we", "can",
    "more", "than", "one",
    "also", "may", "been"
}


english_scores = []

for sample in samples:

    words = re.findall(
        r"\b[a-zA-Z]+\b",
        sample["text"].lower()
    )

    if len(words) == 0:

        score = 0

    else:

        common = sum(
            word in COMMON_ENGLISH
            for word in words
        )

        score = (
            common /
            len(words)
        )

    sample["english_heuristic"] = score

    english_scores.append(
        score
    )


# ============================================================
# PART 8
# PRINT OVERALL RESULTS
# ============================================================

print("\n")
print("=" * 80)
print("CONTENT AUDIT RESULTS")
print("=" * 80)

n = len(samples)

print(
    f"Records examined       : {n:,}"
)

print(
    f"Clean records          : "
    f"{quality_counts['CLEAN']:,} "
    f"({quality_counts['CLEAN']/n*100:.2f}%)"
)

print(
    f"Flagged records        : "
    f"{quality_counts['FLAGGED']:,} "
    f"({quality_counts['FLAGGED']/n*100:.2f}%)"
)

print("\nFlag frequencies:")

for key, value in quality_counts.most_common():

    if key in {
        "CLEAN",
        "FLAGGED"
    }:
        continue

    print(
        f"{key:30s} "
        f"{value:6,} "
        f"({value/n*100:6.2f}%)"
    )


# ============================================================
# PART 9
# DOCUMENT TYPE DISTRIBUTION
# ============================================================

print("\n")
print("=" * 80)
print("DOCUMENT TYPE DISTRIBUTION")
print("=" * 80)

type_counts = Counter(
    s["document_type"]
    for s in samples
)

for doc_type, count in (
    type_counts.most_common()
):

    print(
        f"{doc_type:25s} "
        f"{count:6,} "
        f"({count/n*100:6.2f}%)"
    )


# ============================================================
# PART 10
# TOKEN LENGTH DISTRIBUTION
# ============================================================

import numpy as np

lengths = np.array(
    token_lengths
)

print("\n")
print("=" * 80)
print("TOKEN LENGTH DISTRIBUTION")
print("=" * 80)

print(
    "Mean   :",
    f"{np.mean(lengths):.1f}"
)

print(
    "Median :",
    f"{np.median(lengths):.1f}"
)

print(
    "P90    :",
    f"{np.percentile(lengths, 90):.1f}"
)

print(
    "P95    :",
    f"{np.percentile(lengths, 95):.1f}"
)

print(
    "P99    :",
    f"{np.percentile(lengths, 99):.1f}"
)

print(
    "Maximum:",
    f"{np.max(lengths):,}"
)


# ============================================================
# PART 11
# ENGLISH HEURISTIC RESULTS
# ============================================================

english_scores = np.array(
    english_scores
)

print("\n")
print("=" * 80)
print("ENGLISH-LIKENESS HEURISTIC")
print("=" * 80)

print(
    "Mean score:",
    f"{np.mean(english_scores):.4f}"
)

print(
    "Median:",
    f"{np.median(english_scores):.4f}"
)

print(
    "Records with score >= 0.05:",
    f"{np.sum(english_scores >= 0.05):,}",
    f"({np.mean(english_scores >= 0.05)*100:.2f}%)"
)

print(
    "Records with score < 0.01:",
    f"{np.sum(english_scores < 0.01):,}",
    f"({np.mean(english_scores < 0.01)*100:.2f}%)"
)


# ============================================================
# PART 12
# SHOW WORST FLAGGED EXAMPLES
# ============================================================

print("\n")
print("=" * 80)
print("MOST FLAGGED EXAMPLES")
print("=" * 80)

worst = sorted(
    samples,
    key=lambda x: x[
        "quality_flag_count"
    ],
    reverse=True
)

for i, sample in enumerate(
    worst[:20],
    1
):

    print("\n" + "-" * 80)

    print(
        f"Example {i}"
    )

    print(
        "Article ID:",
        sample["article_id"]
    )

    print(
        "Document type:",
        sample["document_type"]
    )

    print(
        "Tokens:",
        sample["token_count"]
    )

    print(
        "Flags:",
        ", ".join(
            sample["quality_flags"]
        )
    )

    text_preview = (
        sample["text"]
        .replace("\n", " ")
        .strip()
    )

    print(
        "Preview:",
        text_preview[:500]
    )


# ============================================================
# PART 13
# SAVE ALL SAMPLES
# ============================================================

samples_file = (
    CONTENT_AUDIT_DIR /
    "content_audit_samples.jsonl"
)

with open(
    samples_file,
    "w",
    encoding="utf-8"
) as f:

    for sample in samples:

        f.write(
            json.dumps(
                sample,
                ensure_ascii=False
            )
            + "\n"
        )


# ============================================================
# PART 14
# SAVE SUMMARY
# ============================================================

summary = {

    "records_examined":
        n,

    "clean_records":
        quality_counts["CLEAN"],

    "flagged_records":
        quality_counts["FLAGGED"],

    "flag_percent":
        quality_counts["FLAGGED"]
        / max(n, 1)
        * 100,

    "quality_flags":
        dict(quality_counts),

    "document_types":
        dict(type_counts),

    "token_length": {

        "mean":
            float(np.mean(lengths)),

        "median":
            float(np.median(lengths)),

        "p90":
            float(np.percentile(
                lengths,
                90
            )),

        "p95":
            float(np.percentile(
                lengths,
                95
            )),

        "p99":
            float(np.percentile(
                lengths,
                99
            )),

        "maximum":
            int(np.max(lengths))
    },

    "english_heuristic": {

        "mean":
            float(np.mean(
                english_scores
            )),

        "median":
            float(np.median(
                english_scores
            )),

        "below_0.01":
            int(np.sum(
                english_scores < 0.01
            )),

        "above_0.05":
            int(np.sum(
                english_scores >= 0.05
            ))
    }
}


summary_file = (
    CONTENT_AUDIT_DIR /
    "content_audit_summary.json"
)

with open(
    summary_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


print("\n")
print("=" * 80)
print("CONTENT AUDIT COMPLETE")
print("=" * 80)

print(
    "Samples saved :",
    samples_file
)

print(
    "Summary saved :",
    summary_file
)

print()
print(
    "IMPORTANT: Automated flags are indicators, "
    "not automatic reasons for deletion."
)

REPRESENTATIVE CONTENT QUALITY AUDIT
Target samples : 2000
Output folder  : /content/drive/MyDrive/SLM_Biomed/audit/content_audit


PASS 1 — COLLECTING RECORD METADATA
[1/18] shard_00000.jsonl.gz
[2/18] shard_00001.jsonl.gz
[3/18] shard_00002.jsonl.gz
[4/18] shard_00003.jsonl.gz
[5/18] shard_00004.jsonl.gz
[6/18] shard_00005.jsonl.gz
[7/18] shard_00006.jsonl.gz
[8/18] shard_00007.jsonl.gz
[9/18] shard_00008.jsonl.gz
[10/18] shard_00009.jsonl.gz
[11/18] shard_00010.jsonl.gz
[12/18] shard_00011.jsonl.gz
[13/18] shard_00012.jsonl.gz
[14/18] shard_00013.jsonl.gz
[15/18] shard_00014.jsonl.gz
[16/18] shard_00015.jsonl.gz
[17/18] shard_00016.jsonl.gz
[18/18] shard_00017.jsonl.gz

Metadata collected: 20,446,322 records
Elapsed: 6.23 minutes


PASS 2 — BUILDING STRATA
Number of strata: 326


PASS 3 — SELECTING ~2,000 REPRESENTATIVE RECORDS
Selected records: 2000


PASS 4 — RETRIEVING SELECTED TEXT
Reading selected records from shard_00000.jsonl.gz
Reading selected records from shard_00001.jsonl